In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pathlib as Path
import seaborn as sns

In [ ]:
df = pd.read_parquet("C:/Users/Mubarak/Pictures/Machine-Learning-Projects/Credit-default-prediction/data/processed/credit_default_v1.parquet")

In [ ]:
#separating splits
"""
- The versioned, hashed data guarantees EDA <--> modeling alignment
- Split separation ensures EDA is done primarily on the training data, and avoids evaluaiton contamination 
"""

In [8]:
train_df = df[df['split']=='train'].drop(columns=['split'])
test_df = df[df['split']=='test'].drop(columns=['split'])

print(f'Train split: {train_df.shape}')
print(f'Test split: {test_df.shape}')


Train split: (24000, 25)
Test split: (6000, 25)


In [9]:
#Target Analysis
default_rate = train_df['default'].mean()
counts = train_df['default'].value_counts()

print("Class distribution:")
print(counts)
print(f"Default rate:{default_rate:.3f}")

Class distribution:
default
0    18691
1     5309
Name: count, dtype: int64
Default rate:0.221


Class distribution:
default
0    18691
1     5309
Name: count, dtype: int64
Default rate:0.221
-----------------------------------
Interpretation:
- Default rate ≈ 22%
- Dataset is moderately imbalanced
- Accuracy is not a valid metric
- Class weighting and threshold tuning will be required
- Confirms need for probabilistic modeling, bacause based on the business nature, missing any of target (FP/FN) is costly, so there no single decision boundary
- Confirms ROC-AUC will be better than accuracy

"" Because default occurs in only ~22% of cases, accuracy is dominated by the majority class and fails to measure model skill. We therefore need probabilistic models that output calibrated risk scores, and ROC-AUC is preferred because it evaluates ranking performance independently of any fixed decision threshold.""

In [ ]:
#Univariate Feature Inspection
num_summary = train_df.describe().T

print(num_summary)

""" We Look for:
- Scale differences (need scaling)
- Data types (scaling/encoding)
- Skewed distributions
- Outliers
"""

Interpretations:
- Binary variables
- Ordinal variables (PAY_0 … PAY_6)
- Nominal categorical variables (EDU, AGE, Marriage)
- Heavy-tailed continuous variables (LIMIT_BAL, BILL_AMT1 … BILL_AMT6, PAY_AMT1 … PAY_AMT6 )
- Ratios implicitly missing

This confirms:
- Need for a ColumnTransformer
- Different preprocessing per feature group
- Pipelines are mandatory (not optional)

In [ ]:
#Target vs Feature Relationships 
""" Key considerations:
- Separation of power: how each feature distinguishes the target outcomes
- Monotonicity: how critical outcomes (default in this case) move with feature 
- Non-linearity and threshold effects: Does risk change sharply after a certain point?
- Stabilty and Noise: Is this relationship stable, or driven by outliers / noise?
"""
#Numerical Features Separation of power analysis
cols = train_df.select_dtypes(include=['int64', 'float64'])
repay_cols = [c for c in train_df.columns if c.startswith("PAY_")]
for i in cols:
    if i not in ['default', 'ID', 'AGE', 'MARRIAGE', 'EDUCATION', 'SEX'] and i not in repay_cols:
        print(f"Power of {i}:")
        print(train_df.groupby("default")[i].mean())
        
"""
Separation power measures how clearly a feature distinguishes different target outcomes,
and it is used to guide feature selection, feature engineering, model choice, and explainability.
"""

In [ ]:
#Categorical Features Separation of power analysis
train_df.groupby("PAY_0")["default"].mean()


In [ ]:
train_df.groupby("EDUCATION")["default"].count()


In [11]:
repay_cols = [c for c in train_df.columns if c.startswith("PAY_")]

train_df[repay_cols + ["default"]].groupby("default").mean()


,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
default,,,,,,,,,,,,
0,-0.210262,-0.303729,-0.318977,-0.357605,-0.392221,-0.408539,6255.599754,6582.407308,5718.519073,5234.298646,5191.868921,5736.769301
1,0.676399,0.463176,0.368431,0.258429,0.158787,0.112262,3398.368054,3406.973630,3445.813147,3227.291015,3284.494255,3445.427953


In [ ]:
#Correlation Analysis (Numeric Only)
corr = train_df.corr(numeric_only=True)

target_corr = corr["default"].sort_values(ascending=False)
print(target_corr.head(10))


Conclusion:
- Default rate ≈ 22% indicates class imbalance 
- Payment history is dominant signal
- Scale differences require normalization
- No missing values
- No obvious leakage
- Logistic regression viable baseline
- Tree-based models likely add value